In [ ]:
# Parameters
SQL_DB_CONN = ""  # ODBC: Driver={ODBC Driver 18 for SQL Server};Server=<host>.database.fabric.microsoft.com,...
SCALE_SIZE = "small"
AUTH_METHOD = "cli"
SEED = 42


## Generate Healthcare Data

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.healthcare import HealthcareDomain

result = Spindle().generate(domain=HealthcareDomain(), scale=SCALE_SIZE, seed=SEED)
print(f"Generated {sum(len(df) for df in result.tables.values()):,} rows")
for t, df in result.tables.items():
    print(f"  {t}: {len(df):,} rows, cols={list(df.columns)[:4]}")


## Apply HIPAA Masking (DataMasker)

In [ ]:
try:
    from sqllocks_spindle.inference import DataMasker
    masker = DataMasker()
    # DataMasker masks PII columns (names, emails, phone, SSN, addresses, etc.)
    for table_name, df in result.tables.items():
        masked_df = masker.mask(df)
        result.tables[table_name] = masked_df
    print("HIPAA masking applied")
except ImportError:
    print("DataMasker not available — skipping masking demo")


In [ ]:
from sqllocks_spindle.fabric.sql_database_writer import FabricSqlDatabaseWriter
if not SQL_DB_CONN:
    raise ValueError("Set SQL_DB_CONN to your Fabric SQL Database ODBC connection string")
writer = FabricSqlDatabaseWriter(SQL_DB_CONN, auth_method=AUTH_METHOD)
writer.write(result, schema_name="dbo", mode="create_insert")
print("Write to SQL Database: DONE")
